In [ ]:

# ============================================================
# PROJECT 7
# GENETIC DATA COMPRESSION AND PATTERN VISUALIZATION SYSTEM
# Using PCA and LDA
# ============================================================

# 1. IMPORT LIBRARIES

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from IPython.display import display, Markdown

print("Libraries imported successfully!")


# ============================================================
# 2. CREATE SAMPLE GENETIC DATASET
# ============================================================

X, y = make_classification(
    n_samples=300,
    n_features=20,
    n_informative=12,
    n_redundant=4,
    n_classes=3,
    n_clusters_per_class=1,
    random_state=42
)

feature_names = [f"Gene_{i+1}" for i in range(20)]

df = pd.DataFrame(X, columns=feature_names)

df["Disorder"] = y

df["Disorder"] = df["Disorder"].map({
    0: "Type A",
    1: "Type B",
    2: "Type C"
})

# Introduce missing values for preprocessing demonstration
df.loc[0:9, "Gene_1"] = np.nan
df.loc[10:19, "Gene_2"] = np.nan

print("\nSample Genetic Dataset:")
display(df.head())

print("Dataset Shape:", df.shape)
print("Original Number of Features:", len(feature_names))


# ============================================================
# 3. DATA PREPROCESSING
# ============================================================

# Separate features and target
X_data = df.drop("Disorder", axis=1)
y_data = df["Disorder"]

# Handle missing values
imputer = SimpleImputer(strategy="mean")

X_imputed = imputer.fit_transform(X_data)

# Encode target labels
encoder = LabelEncoder()

y_encoded = encoder.fit_transform(y_data)

# Standardize numerical features
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_imputed)

print("\nData Preprocessing Completed!")
print("Missing values handled")
print("Categorical labels encoded")
print("Numerical features standardized")


# ============================================================
# 4. APPLY PCA
# ============================================================

pca_full = PCA()

X_pca_full = pca_full.fit_transform(X_scaled)

explained_variance = pca_full.explained_variance_ratio_

cumulative_variance = np.cumsum(explained_variance)

# Select components preserving at least 95% variance
n_components = np.argmax(cumulative_variance >= 0.95) + 1

pca = PCA(n_components=n_components)

X_pca = pca.fit_transform(X_scaled)

print("\nPCA RESULTS")
print("---------------------------")
print("Original Features:", X_scaled.shape[1])
print("Reduced Components:", n_components)
print("Variance Preserved:",
      round(sum(pca.explained_variance_ratio_) * 100, 2), "%")


# ============================================================
# 5. PCA EXPLAINED VARIANCE VISUALIZATION
# ============================================================

plt.figure(figsize=(10, 5))

plt.plot(
    range(1, len(cumulative_variance) + 1),
    cumulative_variance * 100,
    marker="o"
)

plt.axhline(
    y=95,
    color="red",
    linestyle="--",
    label="95% Variance"
)

plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance (%)")

plt.title("PCA Explained Variance Analysis")

plt.legend()
plt.grid(True)

plt.show()


# ============================================================
# 6. 2D PCA VISUALIZATION
# ============================================================

pca_2d = PCA(n_components=2)

X_pca_2d = pca_2d.fit_transform(X_scaled)

pca_df = pd.DataFrame({
    "PC1": X_pca_2d[:, 0],
    "PC2": X_pca_2d[:, 1],
    "Disorder": y_data
})

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="Disorder",
    palette="Set1",
    s=80
)

plt.title("2D PCA Genetic Pattern Visualization")

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")

plt.grid(True)
plt.show()


# ============================================================
# 7. APPLY LDA
# ============================================================

lda = LinearDiscriminantAnalysis(n_components=2)

X_lda = lda.fit_transform(X_scaled, y_encoded)

lda_df = pd.DataFrame({
    "LD1": X_lda[:, 0],
    "LD2": X_lda[:, 1],
    "Disorder": y_data
})

print("\nLDA Completed Successfully!")


# ============================================================
# 8. 2D LDA VISUALIZATION
# ============================================================

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=lda_df,
    x="LD1",
    y="LD2",
    hue="Disorder",
    palette="Set1",
    s=80
)

plt.title("2D LDA Genetic Pattern Visualization")

plt.xlabel("Linear Discriminant 1")
plt.ylabel("Linear Discriminant 2")

plt.grid(True)
plt.show()


# ============================================================
# 9. PCA VS LDA COMPARISON
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="Disorder",
    palette="Set1",
    ax=axes[0]
)

axes[0].set_title("PCA Visualization")

sns.scatterplot(
    data=lda_df,
    x="LD1",
    y="LD2",
    hue="Disorder",
    palette="Set1",
    ax=axes[1]
)

axes[1].set_title("LDA Visualization")

plt.suptitle("PCA vs LDA Comparison")

plt.tight_layout()
plt.show()


# ============================================================
# 10. CLASSIFICATION USING LDA
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

lda_model = LinearDiscriminantAnalysis()

lda_model.fit(X_train, y_train)

y_pred = lda_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("\nCLASSIFICATION RESULTS")
print("---------------------------")
print("LDA Classification Accuracy:",
      round(accuracy * 100, 2), "%")


# ============================================================
# 11. ANALYTICAL DASHBOARD
# ============================================================

display(Markdown("# Genetic Data Analytical Dashboard"))

display(Markdown("## Dataset Information"))

print("Total Patients:", len(df))
print("Original Genetic Features:", len(feature_names))
print("PCA Components:", n_components)
print("PCA Variance Preserved:",
      round(sum(pca.explained_variance_ratio_) * 100, 2), "%")

display(Markdown("## Disorder Distribution"))

print(df["Disorder"].value_counts())

plt.figure(figsize=(7, 4))

sns.countplot(
    data=df,
    x="Disorder",
    hue="Disorder",
    palette="Set2",
    legend=False
)

plt.title("Genetic Disorder Distribution")

plt.show()


# ============================================================
# 12. DOWNLOAD PROCESSED DATA
# ============================================================

processed_df = pd.DataFrame(
    X_pca,
    columns=[f"PC{i+1}" for i in range(n_components)]
)

processed_df["Disorder"] = y_data.values

processed_df.to_csv(
    "compressed_genetic_data.csv",
    index=False
)

print("\nCompressed Dataset Saved Successfully!")

from google.colab import files

files.download("compressed_genetic_data.csv")

print("\nPROJECT COMPLETED SUCCESSFULLY!")